<a href="https://colab.research.google.com/github/snoopdragon1805/SDC/blob/main/finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Fine-Tuning GPT-2 on Product Reviews in Google Colab

# Install required libraries
#!pip install transformers datasets torch

# Import necessary modules
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import torch
from torch.utils.data import Dataset as TorchDataset
import pandas as pd
import random

# Step 1: Prepare Dataset
# Sample product review data (replace with your own dataset)
reviews = [
    "This product is amazing! The quality exceeded my expectations and it arrived quickly.",
    "Not worth the price. The item broke after just two uses and customer service was unhelpful.",
    "Good value for money. It does what it's supposed to do without any frills.",
    "The packaging was damaged when it arrived, but the product itself works perfectly fine.",
    "I absolutely love this! It's become an essential part of my daily routine.",
    "Decent product for the price, though it could be more durable.",
    "The instructions were unclear but after figuring it out, the product works well.",
    "Would buy again! This has solved a problem I've had for years.",
    "Terrible experience. The product didn't match the description at all.",
    "Fast shipping and excellent customer service. The product itself is just okay."
]

# Create a pandas DataFrame
df = pd.DataFrame({'review': reviews})

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split dataset into train and test
dataset = dataset.train_test_split(test_size=0.2)

# Step 2: Load and Prepare Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Tokenize function
def tokenize_function(examples):
    return tokenizer(examples["review"], truncation=True, padding="max_length", max_length=64)

# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Format for language modeling
lm_datasets = tokenized_datasets.map(
    lambda examples: {
        "input_ids": examples["input_ids"],
        "labels": examples["input_ids"],  # Targets are the same as inputs for causal LM
    },
    batched=True,
    remove_columns=["review"]
)

# Step 3: Load GPT-2 Model
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))

# Step 4: Set Up Training Arguments
training_args = TrainingArguments(
         output_dir="./gpt2-product-reviews",
         overwrite_output_dir=True,
         num_train_epochs=10,
         per_device_train_batch_size=4,
         save_steps=10_000,
         save_total_limit=2,
         prediction_loss_only=True,
         logging_steps=100,
         eval_steps=500,
         report_to="none",
         # Add run_name here
         run_name="my-product-review-finetuning"  # Choose a descriptive name
     )

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["test"],
)

# Step 5: Fine-Tune the Model
print("Starting training...")
trainer.train()
print("Training completed!")

# Save the fine-tuned model
model.save_pretrained("./gpt2-product-reviews-final")
tokenizer.save_pretrained("./gpt2-product-reviews-final")
print("Model saved to './gpt2-product-reviews-final'")

# Step 6: Test the Fine-Tuned Model
def generate_review(prompt, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt",padding=True)
    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test with different prompts
print("\nGenerated Reviews with Fine-Tuned Model:")
print("1:", generate_review("This product is"))
print("2:", generate_review("I would not recommend"))
print("3:", generate_review("The quality of this"))
print("4:", generate_review("After using this for a week"))

# Step 7: Compare with Base GPT-2
base_model = GPT2LMHeadModel.from_pretrained("gpt2")
base_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

def base_generate_review(prompt, max_length=50):
    inputs = base_tokenizer(prompt, return_tensors="pt")
    outputs = base_model.generate(
        inputs.input_ids,
        max_length=max_length,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )
    return base_tokenizer.decode(outputs[0], skip_special_tokens=True)

# Comparison
prompt = "This product is"
print("\nComparison for prompt:", prompt)
print("Fine-tuned:", generate_review(prompt))
print("Base GPT-2:", base_generate_review(prompt))

# Additional Helper Functions
def load_custom_dataset(file_path):
    """Load a custom dataset from CSV file"""
    df = pd.read_csv(file_path)
    return Dataset.from_pandas(df)

def evaluate_model(model, tokenizer, test_samples=5):
    """Evaluate the model on sample prompts"""
    prompts = [
        "This product is",
        "I would recommend",
        "The worst part is",
        "After using this for",
        "The best feature is"
    ]
    for prompt in prompts[:test_samples]:
        print(f"\nPrompt: '{prompt}'")
        print("Generated:", generate_review(prompt))

# Uncomment to use these functions
# dataset = load_custom_dataset("your_reviews.csv")
# evaluate_model(model, tokenizer)

print("\nFine-tuning and evaluation complete!")

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Starting training...


...

3: The quality of this product is amazing.
4: After using this for a week, I've come to like it!


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Comparison for prompt: This product is


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Fine-tuned: This product is made by a professional.
Base GPT-2: This product is not endorsed or sold by Google, nor does it necessarily represent the views of the company or any of its affiliates. Google may have its own policy on this product.

About Google
, founded by former Google senior vice president of

Fine-tuning and evaluation complete!
